# 04 · Export + CPU inference

Exports a run into a self-contained bundle (`export/<name>/`: weights with EMA, tokenizer, ة lexicon, config, default voice) and runs it **on the CPU** with the stock pocket-tts streaming engine. The GPU is not used here.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

REPO = Path("/workspace/EgyPocket-TTS")          # where you cloned the repo
os.environ.setdefault("EGY_DATA", "/workspace/egypocket_data")
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")     # models + HF token persist on the volume
DATA = Path(os.environ["EGY_DATA"])
CATT_DIR = Path("/workspace/catt_tashkeel")      # the uploaded CATT folder
os.chdir(REPO)
sys.path.insert(0, str(REPO))
PY = sys.executable                              # the egypocket conda env
print("python:", PY)
print("repo  :", REPO)
print("data  :", DATA)

In [ ]:
RUN = DATA / "runs/distill_6l"          # the student; use runs/teacher_24l with LAYERS = 24 to hear the teacher
LAYERS = 6
NAME = "egypocket_6l"
CHECKPOINT = None                       # None = newest checkpoint of the run
cmd = f"{PY} -m egypocket.cli export --run {RUN} --name {NAME} --layers {LAYERS}"
if CHECKPOINT: cmd += f" --checkpoint {CHECKPOINT}"
!{cmd}
BUNDLE = DATA / "export" / NAME
!ls -la {BUNDLE}

## Load on CPU

Pocket TTS targets 2 CPU cores; `num_threads` controls it (RunPod CPUs are server cores, a laptop gives comparable numbers).

In [ ]:
import torch
from IPython.display import Audio, display
from egypocket.inference import EgyPocketTTS

torch.set_num_threads(2)
tts = EgyPocketTTS(BUNDLE, num_threads=2)
_ = tts.generate("اهلا.")   # warm-up
def say(text, voice=None):
    audio = tts.generate(text, voice)
    s = tts.last_stats
    print(f"{tts.normalize(text)}\n  {s.audio_sec:.2f}s audio in {s.wall_sec:.2f}s -> x{s.real_time_factor:.1f} real time, first audio after {s.first_chunk_sec*1000:.0f} ms")
    display(Audio(audio, rate=tts.sample_rate))
    return audio

In [ ]:
for text in [
    "النهارده الجو حلو قوي, وانا رايح الشغل بدري علشان الحق الاجتماع.",
    "لو عندك اي سؤال, ابعتلي رسالة وانا هرد عليك في اقرب وقت.",
    "المصريين القدماء بنوا الاهرامات من اكتر من اربع تلاف وخمسمية سنة.",
    "الاجتماع هيبدأ الساعة 3:30 وهيخلص على 5.",
]:
    say(text)

## Diacritics as an override

Write diacritics only where the model gets a word wrong. Final **ة**: plain `ة` → *-a* (pausal), `ةْ` (or `ةَ` `ةِ` `ةُ`) → the *t* is pronounced.

In [ ]:
for text in ["رحت المدينة امبارح.", "رحت المدينةْ امبارح.", "دي مدينةَ جميلة.", "في مدينةِ القاهرة.", "مدينةُ الاسكندرية."]:
    say(text)

In [ ]:
for text in ["العلم نور.", "العِلْم نور.", "العَلَم رمز البلد.",
             "قلبي معاك.", "قَلْبي معاك.", "قالب الكيكة.", "قَالِب الكيكة."]:
    say(text)

## Streaming (first audio chunk latency)

In [ ]:
import time, numpy as np
text = "ولو ركزنا شوية في الحكاية دي, هنلاقي ان المصريين القدماء كانوا سابقين عصرهم في حاجات كتير. وده اللي هنتكلم عنه النهارده."
t0 = time.monotonic(); chunks = []
for i, chunk in enumerate(tts.stream(text)):
    if i == 0: print(f"first chunk after {(time.monotonic()-t0)*1000:.0f} ms ({len(chunk)/tts.sample_rate*1000:.0f} ms of audio)")
    chunks.append(chunk)
s = tts.last_stats; print(f"{s.audio_sec:.1f}s audio in {s.wall_sec:.1f}s -> x{s.real_time_factor:.1f}")
Audio(np.concatenate(chunks), rate=tts.sample_rate)

## Voice prompt

The default voice is a held-out clip of the training speaker (`voice.wav`). You can pass any wav as `voice=` (it is encoded once and cached) or export it to a `.safetensors` voice state that loads instantly. This first model is single-speaker, so other voices are not expected to clone well yet.

In [ ]:
display(Audio(str(BUNDLE / "voice.wav")))
tts.export_voice(BUNDLE / "voice.wav", BUNDLE / "voice_copy.safetensors")
say("الصوت ده جاي من ملف محفوظ.", voice=BUNDLE / "voice_copy.safetensors")

## int8 dynamic quantization

Smaller weights; whether it is faster depends on the CPU (compare the two real-time factors and keep the faster setting).

In [ ]:
tts_q = EgyPocketTTS(BUNDLE, num_threads=2, quantize=True)
_ = tts_q.generate("اهلا.")
audio = tts_q.generate("النهارده الجو حلو قوي, وانا رايح الشغل بدري.")
s = tts_q.last_stats; print(f"int8: x{s.real_time_factor:.1f} real time, first audio {s.first_chunk_sec*1000:.0f} ms")
Audio(audio, rate=tts_q.sample_rate)

## Test-set sentences (never seen in training)

In [ ]:
from egypocket.data.jsonl import read_jsonl
test = read_jsonl(DATA / "work/manifest_test.jsonl")
for r in test[:3]:
    say(r["text"][:200])

## Package the bundle for download (Jupyter file browser → right click → Download)

In [ ]:
import shutil
archive = shutil.make_archive(str(DATA / "export" / NAME), "zip", BUNDLE)
print(archive)

Use it anywhere with a CPU:

```python
from egypocket.inference import EgyPocketTTS
tts = EgyPocketTTS("egypocket_6l")        # the unzipped folder
tts.save("النهارده الجو حلو قوي.", "out.wav")
```